In [79]:
import os
from huggingface_hub import InferenceClient
from tools import create_calender_event
import json

In [80]:
client=InferenceClient(
    api_key=os.getenv("HF_API_KEY")
)

messages=[]

calender_tool={
    "type" : "function",
    "function" : {
        "name" : "create_calender_event",
        "description" : "create a calender event",
        "parameters" : {
            "title" : {
                "type" : "string",
                "description" : "The title of the event"
            },
            "date" : {
                "type" : "string",
                "description" : "The date at which the event need to be scheduled"
            },
            "time" : {
                "type" : "string",
                "description" : "The time at which the event needs to be scheduled"
            }
        }
    }
}

tool_registery={
    "create_calender_event" : create_calender_event
}

In [81]:
messages.append({
    "role":"system",
    "content" : "You are a helpful AI assistant and your task is to answer questions without hallucinations."
})

available_tools=[calender_tool]

user_input=input(">>")
print(f">> {user_input}")

messages.append(
    {
        "role" : "user",
        "content" : user_input
    }
)

while True:
    response=client.chat.completions.create(
        model="meta-llama/Llama-3.1-8B-Instruct",
        messages=messages,
        tools=available_tools
    )

    assistant_response=response.choices[0].message
    
    if assistant_response.tool_calls:
        tool_call_list=[]
        for tool_call in assistant_response.tool_calls:
            tool_name=tool_call.function.name
            tool_args=tool_call.function.arguments

            tool_call_list.append({
                "id" : tool_call.id,
                "type" : "function",
                "function" : {
                    "name" : tool_name,
                    "arguments" : tool_args
                }
            })

        messages.append({
            "role":"assistant",
            "content" : None,
            "tool_calls" : tool_call_list
        })


        for tool_call in assistant_response.tool_calls:
            tool_name=tool_call.function.name
            tool_args=tool_call.function.arguments

            tool=tool_registery[tool_name]
            tool_res=tool(**json.loads(tool_args))

            messages.append({
                "role" : "tool",
                "content" : json.dumps(tool_res),
                "tool_call_id" : tool_call.id
            })

    else:
        print(f"LLM >> {assistant_response.content}")
        break

>> hey , schedule a meet with elon this monday at 10 am in my calender
LLM >> I've scheduled the meeting with Elon in your calendar.
